# 2.9 — pretrained backbones as fixed feature extractors

**Self-contained.** The model, its head, and all three configs are defined in this notebook
and injected at runtime — nothing needs to be pulled, edited, or added to the package. It
does use the repo's data pipeline, splits, training loop and metrics, because results that
came from different code would not be comparable to v27/v28.

Three arms, ~15–40 min each:

| arm | backbone | input encoding | trained parameters |
|---|---|---|---|
| `05` | ResNet18, frozen | `one_hot` (3 indicator planes) | head only |
| `06` | ResNet18, frozen | `grayscale_rgb` + ImageNet norm | head only |
| `07` | MobileNetV3-Small, frozen | `grayscale_rgb` + ImageNet norm | head only |

## The two questions

**05 vs 06 — how should a wafer map be fed to an ImageNet model?** `one_hot` keeps all
three states (outside wafer / functional die / defective die) on separate channels but is
nothing like a photograph, so pretrained filters meet a distribution they never saw.
`grayscale_rgb` collapses the states onto one axis — where "defective" is merely twice
"functional" — but places the input where the weights were fitted. Phase 2 measured
`one_hot` as *better* when training from scratch (0.8173 vs 0.8042); with frozen weights
the pressure runs the other way.

**06 vs 07 — is it pretrained features, or ResNet18 specifically?** Same encoding, a
backbone 20x smaller.

## Why frozen and not fine-tuned

The brief asks for a meaningful component of our own. A frozen backbone makes every trained
parameter ours: the pretrained half contributes fixed representations, the MLP head is the
model. It is also the honest comparison — fine-tuning 11M ImageNet parameters on 121k wafer
maps mostly measures how much capacity you can throw at the problem, not what transfer buys.

To fine-tune anyway, add to `OVERRIDES` in section 4:
`"model.kwargs.freeze_encoder=false"` — and expect it to take far longer.

## What to expect

Probably a loss. The best from-scratch model on this data is 0.8900 with 2.83M parameters,
and 0.8883 with 414k. ImageNet features are edges, textures and object parts; a wafer map is
a sparse binary pattern on a disc. **A clear negative is a good slide** — it is the direct
evidence for having built models from scratch, and it is the comparison the brief asks for.

## 1. Setup

Mounts Drive, copies the dataset, installs the package. Every step is a no-op if already done.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

Run `wandb login` in a terminal first if this reports 'not authenticated'.

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")

## 3. The model — defined here, not imported

Read the docstring: the `train()` override is not decoration. Without it the "frozen"
backbone's BatchNorm layers keep updating their running statistics from wafer maps every
epoch, so the encoder's outputs drift while its weights sit still, and the experiment
silently stops being a frozen-feature experiment. The cell asserts the freeze holds.

In [ ]:
"""Frozen pretrained backbone + a small MLP head, defined here so this
notebook runs on a checkout that has not been pulled.

The identical class also lives at `src/fdl_project/models/frozen_backbone.py`
with tests in `tests/test_frozen_backbone.py`; registering it here simply
overwrites the registry entry with the same thing.

Two details that are easy to get wrong and that both matter here:

1. `requires_grad = False` alone does NOT freeze a backbone that contains
   BatchNorm. `fit_model` calls `model.train()` every epoch, which puts BN back
   into training mode, and BN then keeps updating `running_mean`/`running_var`
   from wafer maps. The encoder's *outputs* drift even though its weights do
   not. Overriding `train()` to force the encoder back to `eval()` is what
   actually makes "frozen" mean frozen.

2. The submodules must be named `encoder` and `head`. Every config in this
   project addresses parameter groups with regexes like `^encoder\.`, and the
   checkpoint metadata assumes that layout.
"""

from torch import Tensor, nn

from fdl_project.constants import NUM_CLASSES
from fdl_project.config.registry import MODEL_REGISTRY
from fdl_project.data.preprocessing import WAFER_STATE_COUNT

BACKBONE_WIDTH = {          # features the backbone emits before its own classifier
    "resnet18": 512,
    "resnet34": 512,
    "mobilenet_v3_small": 576,
    "mobilenet_v3_large": 960,
    "efficientnet_b0": 1280,
}


class FrozenBackboneMLP(nn.Module):
    """A torchvision backbone as a fixed feature extractor, plus our own MLP.

    This is the "extract pretrained features and train original layers on top"
    strategy: the pretrained half contributes representations it learned from
    photographs, and every trained parameter in the model is ours.
    """

    def __init__(
        self,
        *,
        architecture: str = "resnet18",
        hidden_features: int = 256,
        dropout: float = 0.3,
        freeze_encoder: bool = True,
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        if architecture not in BACKBONE_WIDTH:
            raise ValueError(
                f"Unsupported architecture {architecture!r}. "
                f"Available: {', '.join(sorted(BACKBONE_WIDTH))}."
            )
        if not 0 <= dropout < 1:
            raise ValueError("dropout must be in [0, 1).")

        import torchvision.models as tv

        weights = None
        if pretrained:
            enum_name = {
                "resnet18": "ResNet18_Weights",
                "resnet34": "ResNet34_Weights",
                "mobilenet_v3_small": "MobileNet_V3_Small_Weights",
                "mobilenet_v3_large": "MobileNet_V3_Large_Weights",
                "efficientnet_b0": "EfficientNet_B0_Weights",
            }[architecture]
            weights = getattr(tv, enum_name).DEFAULT
        backbone = getattr(tv, architecture)(weights=weights)

        width = BACKBONE_WIDTH[architecture]
        if architecture.startswith("resnet"):
            backbone.fc = nn.Identity()
        else:
            # mobilenet/efficientnet end in a Sequential classifier; dropping it
            # leaves features + avgpool + flatten, which emits `width` values.
            backbone.classifier = nn.Identity()

        self.architecture = architecture
        self.frozen = freeze_encoder
        self.encoder = backbone
        # The "original neural layers" the project brief asks for: one hidden
        # layer, not a bare linear probe, so the head can recombine ImageNet
        # features rather than only reweight them.
        self.head = nn.Sequential(
            nn.Linear(width, hidden_features),
            nn.BatchNorm1d(hidden_features),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_features, NUM_CLASSES),
        )
        if freeze_encoder:
            for parameter in self.encoder.parameters():
                parameter.requires_grad = False
            self.encoder.eval()

    def train(self, mode: bool = True) -> "FrozenBackboneMLP":
        """Keep a frozen encoder in eval mode however the loop calls train()."""

        super().train(mode)
        if self.frozen:
            self.encoder.eval()
        return self

    def forward(self, inputs: Tensor) -> Tensor:
        if inputs.ndim != 4 or inputs.shape[1] != WAFER_STATE_COUNT:
            raise ValueError(
                "FrozenBackboneMLP inputs must have shape "
                f"(batch, {WAFER_STATE_COUNT}, height, width); "
                f"got {tuple(inputs.shape)}."
            )
        if not inputs.is_floating_point():
            raise TypeError("FrozenBackboneMLP inputs must be floating-point.")
        return self.head(self.encoder(inputs))


MODEL_REGISTRY["frozen_backbone_mlp"] = lambda **kwargs: FrozenBackboneMLP(**kwargs)
print("registered 'frozen_backbone_mlp'")

# Prove the freeze holds through a train() call -- the bug this class exists to
# avoid is silent, so it is checked rather than assumed.
_probe = FrozenBackboneMLP(architecture="resnet18", pretrained=False)
_probe.train()
_encoder_training = any(m.training for m in _probe.encoder.modules())
_trainable = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
_total = sum(p.numel() for p in _probe.parameters())
assert not _encoder_training, "encoder went back into train mode -- BN stats would drift"
assert all(not p.requires_grad for p in _probe.encoder.parameters())
print(f"  resnet18 probe: {_trainable:,} trainable of {_total:,} total, encoder stays eval")
del _probe

## 4. The three configs — written from this notebook

Everything unnamed is inherited from `configs/train/defaults.yaml`: letterbox geometry,
rotation p=0.5, inverse-sqrt sampler, cross-entropy, AdamW 7e-4, batch 256, fp16 AMP,
patience 10.

128x128, not 64: ResNet18 downsamples by 32, so a 64x64 input leaves a 2x2 feature map to
pool over — too little spatial extent for the backbone to say anything. At 128 it is 4x4.
128 is also the largest size that still fits the 2 GiB geometry cache.

In [ ]:
"""The three arms, written to disk from here so the notebook carries them.

Everything not named below is inherited from configs/train/defaults.yaml, which
is the pipeline settled in phase 2 and confirmed in v26/v27:

    letterbox geometry, rotation p=0.5 +/-180deg, inverse-sqrt weighted sampler,
    cross-entropy, AdamW 7e-4 / wd 1e-4, batch 256, fp16 AMP, patience 10.

The one thing that varies across arms besides the backbone is how a wafer map
is turned into three channels, and that is the actual experiment:

  one_hot        channel 0 = outside wafer, 1 = functional die, 2 = defective die.
                 Three indicator planes. Nothing about it resembles a photograph,
                 so ImageNet's first-layer filters are being applied to a
                 distribution they never saw. Normalization must be 'none' --
                 the schema rejects ImageNet statistics on indicator channels,
                 because that arithmetic has no meaning.

  grayscale_rgb  the categorical map scaled to [0, 1] and repeated across R, G, B,
                 then standardised with ImageNet mean/std. This is the standard
                 way to feed a single-channel image to an ImageNet backbone, and
                 it puts the input in the distribution the weights were fitted on
                 -- at the cost of collapsing three distinct states onto one axis
                 where "defective" is merely twice "functional".

Arm 05 vs 06 is exactly that trade: information-preserving but out-of-distribution,
against lossy but in-distribution. Phase 2 measured grayscale as *worse* than
one-hot when training from scratch (0.8042 vs 0.8173) -- with frozen ImageNet
weights the pressure runs the other way, which is why it is worth one run.
"""

import textwrap
from pathlib import Path

SERIES = "v29_pretrained"
CONFIG_DIRECTORY = REPO / "configs/train" / SERIES

# The repo carries the same three configs. If the checkout already has them,
# use those and write nothing -- the version under git is the one of record.
# Writing them from here is the fallback that keeps this notebook runnable on a
# checkout that predates them.
WRITE_CONFIGS = not any(CONFIG_DIRECTORY.glob("*.yaml"))
CONFIG_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("configs:", "writing from this notebook" if WRITE_CONFIGS else "already in the repo")

SHARED = """
data:
  preprocessing:
    target_size: [128, 128]
  augmentation:
    name: rotation
    probability: 0.5
    kwargs: {degrees: 180.0}

imbalance:
  preset: inverse_sqrt_sampler

trainer:
  # A frozen backbone trains only the head, so it converges in a fraction of the
  # epochs a from-scratch model needs. Patience 10 still decides when to stop.
  max_epochs: 60
"""

ARMS = {
    "05_resnet18_frozen_onehot.yaml": """
# Frozen ImageNet ResNet18 reading our one-hot channels directly.
# Keeps every bit of the categorical encoding; asks ImageNet filters to make
# sense of an input unlike anything they were trained on.
name: v29-resnet18_frozen_onehot

model:
  name: frozen_backbone_mlp
  kwargs: {architecture: resnet18, hidden_features: 256, dropout: 0.3}
""" + SHARED + """
    # one_hot forbids any normalization: ImageNet statistics on indicator
    # channels is arithmetic without meaning, and the schema enforces it.
""",
    "06_resnet18_frozen_grayscale.yaml": """
# The same frozen backbone, fed the way an ImageNet model expects: one grey
# channel repeated to RGB and standardised with ImageNet mean/std.
# Loses the categorical distinction, gains distribution match. 05 vs 06 is the
# whole question.
name: v29-resnet18_frozen_grayscale

model:
  name: frozen_backbone_mlp
  kwargs: {architecture: resnet18, hidden_features: 256, dropout: 0.3}
""" + SHARED,
    "07_mobilenet_frozen_grayscale.yaml": """
# A second frozen backbone on the better-performing encoding, to separate
# "pretrained features help" from "ResNet18's features help".
# mobilenet_v3_small is 20x smaller than ResNet18 and emits 576 features.
name: v29-mobilenet_frozen_grayscale

model:
  name: frozen_backbone_mlp
  kwargs: {architecture: mobilenet_v3_small, hidden_features: 256, dropout: 0.3}
""" + SHARED,
}

# The two grayscale arms need the encoding block appended; the one-hot arm
# inherits the default encoding untouched.
GRAYSCALE = """    encoding: grayscale_rgb
    normalization: imagenet
"""
if WRITE_CONFIGS:
    for filename, body in ARMS.items():
        text = body
        if "grayscale" in filename:
            text = text.replace(
                "    target_size: [128, 128]\n",
                "    target_size: [128, 128]\n" + GRAYSCALE,
            )
        (CONFIG_DIRECTORY / filename).write_text(text.lstrip("\n"))

for path in sorted(CONFIG_DIRECTORY.glob("*.yaml")):
    print(path.name)

## 5. Run

Cheapest-first, resumable between arms via `results.csv` and within an arm via Drive checkpoints.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

RERUN = False
BUDGET_HOURS = 3.0

CONFIGS = sorted(CONFIG_DIRECTORY.glob("*.yaml"))
OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT / "results.csv"

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},pretrained]"]

results = []
if RESULTS_CSV.exists() and not RERUN:
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"resuming: {len(results)} arm(s) done")

done = {r["run"] for r in results}
dataframe = load_wm811k_dataframe(DATASET)
session_started = time.monotonic()

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    assert config.trainer.batch_size == 256, "defaults.yaml was not inherited"
    assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"

    if config.name in done:
        print(f"skip  {config.name}")
        continue
    elapsed = (time.monotonic() - session_started) / 3600
    if BUDGET_HOURS is not None and elapsed > BUDGET_HOURS:
        print(f"\nbudget reached ({elapsed:.1f} h) -- stopping before {config.name}")
        break

    model = build_model(config.model.name, **config.model.kwargs)
    trainable = count_trainable_parameters(model)
    total = sum(p.numel() for p in model.parameters())
    encoding = config.data.preprocessing.encoding
    print(f"\n=== {config.name}")
    print(f"    {config.model.kwargs['architecture']}, {encoding}/"
          f"{config.data.preprocessing.normalization}, "
          f"{trainable:,} trainable of {total:,}")
    del model

    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "run": config.name,
        "backbone": config.model.kwargs["architecture"],
        "encoding": encoding,
        "normalization": config.data.preprocessing.normalization,
        "trainable": trainable,
        "total": total,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    if HAS_DRIVE:
        shutil.copy2(RESULTS_CSV, DRIVE / f"{SERIES}_results.csv")
    row = results[-1]
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min")

print(f"\n{len(results)}/{len(CONFIGS)} arms complete -> {RESULTS_CSV}")

## 6. Read the result

Judged against the from-scratch numbers already measured on these exact splits, not against
an arm inside this notebook.

The noise floor is **0.02** — `baseline_cnn` on one fixed config scored 0.8800, 0.8696 and
0.8646 across three runs. Do not rank two arms that sit closer than that.

In [ ]:
# From-scratch reference points, measured on the same splits and pipeline.
FROM_SCRATCH = {
    "baseline_cnn (157k, 64px)":   0.8646,
    "convnext_style (414k, 64px)": 0.8883,
    "resnet_style (2.83M, 64px)":  0.8900,
}
BEST_FROM_SCRATCH = max(FROM_SCRATCH.values())
NOISE_FLOOR = 0.02      # baseline_cnn measured 3x: 0.8800 / 0.8696 / 0.8646

frame = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
frame["vs_best_scratch"] = (frame["macro_f1"] - BEST_FROM_SCRATCH).round(4)
frame["trainable_pct"] = (100 * frame["trainable"] / frame["total"]).round(1)
pd.set_option("display.width", 240)
display(frame)

print("From scratch, for comparison:")
for label, score in FROM_SCRATCH.items():
    print(f"  {label:30} {score:.4f}")

if {"v29-resnet18_frozen_onehot", "v29-resnet18_frozen_grayscale"} <= set(frame["run"]):
    scored = frame.set_index("run")["macro_f1"]
    delta = scored["v29-resnet18_frozen_grayscale"] - scored["v29-resnet18_frozen_onehot"]
    verdict = "REAL" if abs(delta) > NOISE_FLOOR else "noise"
    print(f"\nEncoding, same frozen backbone: grayscale+imagenet minus one_hot "
          f"= {delta:+.4f}  [{verdict}]")
    print("  Positive means matching ImageNet's input distribution beat keeping")
    print("  the categorical states. Negative means the opposite, and phase 2's")
    print("  from-scratch finding (one_hot > grayscale) carries over.")

if {"v29-resnet18_frozen_grayscale", "v29-mobilenet_frozen_grayscale"} <= set(frame["run"]):
    scored = frame.set_index("run")["macro_f1"]
    delta = scored["v29-mobilenet_frozen_grayscale"] - scored["v29-resnet18_frozen_grayscale"]
    print(f"\nBackbone, same encoding: mobilenet_v3_small minus resnet18 "
          f"= {delta:+.4f}")

gap = BEST_FROM_SCRATCH - frame["macro_f1"].max()
print(f"\nBest from-scratch minus best frozen-pretrained: {gap:+.4f}")
print("A large positive gap is the expected and reportable result: ImageNet")
print("features transfer poorly to categorical wafer maps, and a 414k CNN")
print("trained from scratch on 121k wafers beats a frozen 11M ResNet18.")

## 7. For the presentation

This notebook produces one slide regardless of which way it lands:

* **If frozen pretrained loses clearly** — that is the justification for the whole
  from-scratch half of the project, stated with a number instead of an assumption. ImageNet
  features are edges, textures and object parts; a wafer map is a sparse binary pattern on a
  disc, and the mismatch is the point.
* **If it wins or ties** — a frozen 11M backbone matching a 414k CNN trained from scratch is
  the more interesting result, and the follow-up is fine-tuning.

Either way report `trainable` alongside `macro_f1`. A head-only arm trains ~150k parameters
against ResNet18's 11.2M total, and that ratio is the argument.